In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Uncomment in case packages are not installed
!pip install timm
!pip install transformers
!pip install pyspellchecker
!pip install torch torchvision
!pip install ftfy
!pip install regex
!pip install sentencepiece
# !pip install git+https://github.com/openai/CLIP.git

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 43.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 35.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 39.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 17.1 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling nvidia-nvjitlink-cu12-12.5.82:
      Successfully uninstalled nvidia-nvjitlink

In [8]:
import torch
# import clip
import argparse
import numpy as np

torch.cuda.empty_cache()
import pandas as pd
import re

import os
import cv2
import gc
import itertools
from tqdm.autonotebook import tqdm
import albumentations as A

from torch import nn
import torch.nn.functional as F
import timm
from transformers import DistilBertModel, DistilBertConfig, DistilBertTokenizer

#NLTK
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords
# This allows to create individual objects from a bog of words
nltk.download('punkt')
from nltk.tokenize import word_tokenize
# Lemmatizer helps to reduce words to the base form
nltk.download('wordnet')
from nltk.stem import WordNetLemmatizer
nltk.download('words')
# Ngrams allows to group words in common pairs or trigrams..etc
from nltk import ngrams
nltk.download('names')
from nltk.corpus import names
# We can use counter to count the objects
from collections import Counter

# from spellchecker import SpellChecker

import spacy

import operator

# This is our visual library
import seaborn as sns
import matplotlib
import matplotlib.pyplot as plt
import math
import seaborn as sns
from wordcloud import WordCloud
from wordcloud import ImageColorGenerator

from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package words to /root/nltk_data...
[nltk_data]   Package words is already up-to-date!
[nltk_data] Downloading package names to /root/nltk_data...
[nltk_data]   Package names is already up-to-date!


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Initialization

In [9]:
class CFG:
    # If there's a GPU available...
    if torch.cuda.is_available():
      # Tell PyTorch to use the GPU.
      device = torch.device("cuda")
      print('There are %d GPU(s) available.' % torch.cuda.device_count())
      print('We will use the GPU:', torch.cuda.get_device_name())
    # If not...
    else:
      print('No GPU available, using the CPU instead.')
      device = torch.device("cpu")

    image_path = "/kaggle/input/mami-dataset/TRAINING/TRAINING/"
    # image_path = "/kaggle/input/mami-dataset/test/test/"

    # train_file = '/content/drive/MyDrive/Experiments/Hate Detection with CLIP/train_small.csv'
    train_file = '/content/drive/MyDrive/MAMI Misogyny using CL/MAMI Dataset/train (2).xls'
    # train_file = '/content/drive/MyDrive/Datasets/Facebook Data/train.jsonl'


    # val_file = '/content/drive/MyDrive/Experiments/Hate Detection with CLIP/validation_small.csv'
    val_file = '/content/drive/MyDrive/MAMI Misogyny using CL/MAMI Dataset/validation (2).csv'
    # val_file = '/content/drive/MyDrive/Datasets/Facebook Data/dev.jsonl'

#     test_small = '/content/drive/MyDrive/Experiments/Hate Detection with CLIP/test_small.csv'
    test_file = '/content/drive/MyDrive/MAMI Misogyny using CL/MAMI Dataset/test (2).csv'


    head_lr = 1e-5
    image_encoder_lr = 1e-5
    text_encoder_lr = 1e-5
    # weight_decay = 1
    weight_decay = 0.1
    patience = 1
    factor = 0.8
    epochs = 10

#     model_name = 'resnet50'
#     image_embedding = 2048 # For resnet50

    model_name = 'ViT'
    image_embedding = 768 # For ViT
    text_encoder_model = "bert-base-uncased"
    text_embedding = 768
    text_tokenizer = "bert-base-uncased"
    max_length = 200 # Max length of captions (text describing the img)

    pretrained = True # for both image encoder and text encoder
    trainable = True # for both image encoder and text encoder
    temperature = 1.0

    # image size
    size = 224 # for resnet50, 'vit_large_patch16_224', 'vit_base_patch16_224'
#     size = 384 # for 'vit_large_patch16_384'

    # for projection head; used for both image and text encoders
    num_projection_layers = 1
    projection_dim = 256
    dropout = 0.1
    # dropout = 0.4


There are 1 GPU(s) available.
We will use the GPU: Tesla T4


In [10]:

import json

#For Loading Images
from PIL import Image

#For displaying loadbar
from tqdm import tqdm

#Importing pytorch to finetune our clip
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

In [11]:
df_train = pd.read_csv(CFG.train_file, header='infer', keep_default_na=False)
# df_train = pd.read_csv(CFG.train_file, header='infer', keep_default_na=False, nrows=4)
df_train = df_train.sample(frac=1).reset_index(drop=True)

# df_train.rename(columns={'Text Transcription': 'transcripts'}, inplace=True)
train_image_names = df_train['file_name']
train_image_path = []
for name in train_image_names:
  train_image_path.append(CFG.image_path + name)

train_texts =  df_train['transcripts']
train_texts = train_texts.astype(str)
train_labels =  df_train['misogynous']


df_val = pd.read_csv(CFG.val_file, header='infer', keep_default_na=False)
df_val = df_val.sample(frac=1).reset_index(drop=True)
# df_val.rename(columns={'Text Transcription': 'transcripts'}, inplace=True)
val_image_names = df_val['file_name']
val_image_path = []
for name in val_image_names:
  val_image_path.append(CFG.image_path + name)

val_texts =  df_val['transcripts']
val_texts = val_texts.astype(str)
val_labels =  df_val['misogynous']

## Load Image and Text Encoders

In [ ]:
from datasets import load_dataset, load_from_disk, Dataset, Features, Array3D
import transformers
from transformers import ViTFeatureExtractor, ViTForImageClassification,ViTImageProcessorFast, AutoImageProcessor, ViTModel, BertModel, BertTokenizer

model_vit = ViTModel.from_pretrained("google/vit-base-patch16-224-in21k")
feature_extractor = ViTImageProcessorFast.from_pretrained("google/vit-base-patch16-224-in21k")

bert_tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
model_bert = BertModel.from_pretrained("bert-base-uncased").to(CFG.device)

config.json:   0%|          | 0.00/502 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/160 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

DistilBERT

# Training

## Setting Up Model and Data loaders

In [ ]:
class CLIPDataset(torch.utils.data.Dataset):
    def __init__(self, dataframe, tokenizer, feature_extractor):
        """
        image_filenames and cpations must have the same length; so, if there are
        multiple captions for each image, the image_filenames must have repetitive
        file names
        """

        self.image_filenames = dataframe['file_name']
        self.captions = list(dataframe['transcripts'])
        self.labels = dataframe['misogynous'] # Comment for testing

        self.encoded_captions = tokenizer.batch_encode_plus(
            list(dataframe['transcripts']),
            padding=True,
            truncation=True,
            max_length=250,
            return_attention_mask=True,
            return_tensors='pt'
        )
#         self.transforms = transforms
        self.feature_extractor = feature_extractor


    def __getitem__(self, idx):
        item = {
            key: torch.tensor(values[idx])
            for key, values in self.encoded_captions.items()
        }

        image = cv2.imread(f"{CFG.image_path}/{self.image_filenames[idx]}")
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
#         image = self.transforms(image=image)['image']
        image = self.feature_extractor(images=image, return_tensors="pt")['pixel_values'].squeeze(0)
#         image = self.feature_extractor(images=image, return_tensors="pt")

#         item['image'] = torch.tensor(image).permute(2, 0, 1).float()
        item['image'] = image
        item['caption'] = self.captions[idx]
        item['label'] = self.labels[idx]
        # item['filename'] = self.image_filenames[idx]

        return item


    def __len__(self):
        return len(self.captions)



def get_transforms(mode="train"):
    if mode == "train":
        return A.Compose(
            [
                A.Resize(CFG.size, CFG.size, always_apply=True),
                A.Normalize(max_pixel_value=255.0, always_apply=True),
            ]
        )
    else:
        return A.Compose(
            [
                A.Resize(CFG.size, CFG.size, always_apply=True),
                A.Normalize(max_pixel_value=255.0, always_apply=True),
            ]
        )


def collate_fn(batch):
    item = {
        key: torch.tensor(values) if key != "caption" else values
        for key, values in batch[0].items()
    }
    return item


def build_loaders(dataframe, tokenizer,feature_extractor, mode):
    transforms = get_transforms(mode=mode)
    dataset = CLIPDataset(
        dataframe,
        tokenizer=tokenizer,
        feature_extractor=feature_extractor,
#         transforms = transforms
    )
    dataloader = torch.utils.data.DataLoader(
        dataset,
        batch_size=16,
#         collate_fn=collate_fn,
        # num_workers=CFG.num_workers,
        # shuffle=False if mode == "train" else False,
        shuffle=False,
    )
    return dataloader

train_dataloader = build_loaders(df_train, bert_tokenizer,feature_extractor, mode="train")
val_dataloader = build_loaders(df_val, bert_tokenizer,feature_extractor, mode="valid")

In [ ]:
class ProjectionHead(nn.Module):
    def __init__(
        self,
        embedding_dim, # Size of input vector  images (2048) and text (768)
        projection_dim=CFG.projection_dim, # Size of output vector : 256
        dropout=CFG.dropout
    ):
        super().__init__()
        self.projection = nn.Linear(embedding_dim, projection_dim)
        self.gelu = nn.GELU()
        self.fc = nn.Linear(projection_dim, projection_dim)
        self.dropout = nn.Dropout(dropout)
        self.layer_norm = nn.LayerNorm(projection_dim)

    def forward(self, x):
        projected = self.projection(x)
        x = self.gelu(projected)
        x = self.fc(x)
        x = self.dropout(x)
        x = x + projected
        x = self.layer_norm(x)
        return x



# Using ViT for Image encoding
class ImageEncoder(nn.Module):
    """
    Encode images to a fixed size vector. In case of ResNet50 the vector size will be 2048
    """
    def __init__(
        self, model_name=CFG.model_name, pretrained=CFG.pretrained, trainable=CFG.trainable
    ):
        super().__init__()
        # self.model = model_vit.to(CFG.device)
        self.model = ViTModel.from_pretrained("google/vit-base-patch16-224-in21k", num_labels=2).to(CFG.device)

        for p in self.model.parameters():
            p.requires_grad = CFG.trainable

    def forward(self, x):
        # Get the outputs from the ViT model
        outputs = self.model(pixel_values=x)

        # The last hidden state has shape [batch_size, num_patches+1, hidden_size]
        # We can take the output corresponding to the [CLS] token
#         last_hidden_state = outputs.hidden_states[-1]
        cls_output = outputs.last_hidden_state[:, 0, :]

        return cls_output


class Identity(nn.Module):
    def __init__(self):
        super(Identity, self).__init__()

    def forward(self, x):
        return x


class TextEncoder(nn.Module):

    #Output hidden representation for each token is a vector with size 768

    def __init__(self, model_name="bert-base-uncased", pretrained=CFG.pretrained, trainable=CFG.trainable):
        super().__init__()

        # self.model = model_bert
        self.model = transformers.BertModel.from_pretrained("bert-base-uncased").to(CFG.device)

        for p in self.model.parameters():
            p.requires_grad = CFG.trainable

        # we are using the CLS token hidden representation as the sentence's embedding
        self.target_token_idx = 0

    def forward(self, input_ids, attention_mask, token_type_ids):
#         _, hidden_state = self.model(input_ids, attention_mask, token_type_ids)
#         return hidden_state

        outputs = self.model(input_ids, attention_mask, token_type_ids)
        cls_embedding = outputs[1]
        return cls_embedding

## Loss Function

In [ ]:
def calculate_loss(cos_sim, labels):

        total_loss = 0
        p_pairs_sim = torch.tensor([]).to(CFG.device)
        all_pairs_sim = torch.tensor([]).to(CFG.device)
        positive_loss = 0
        negative_loss = 0
        sum = 0
        temperature = 0.4
        # logits_per_text = torch.zeros_like(cos_sim)
        for i in range(len(labels)):
            for j in range(i+1, len(labels)):
                all_pairs_sim = torch.cat((all_pairs_sim, cos_sim[i, j].unsqueeze(0)), dim=0)
                if labels[i] == labels[j]:
                    p_pairs_sim = torch.cat((p_pairs_sim, cos_sim[i, j].unsqueeze(0)), dim=0)

        # print("all_pairs_sim", all_pairs_sim)
        # Calculate the denominator for normalization
        exponentiated_sim = torch.exp(all_pairs_sim/temperature)

        # exponentiated_sim = all_pairs_sim
        denominator = torch.sum(exponentiated_sim, dim=0)
       # print("Denominator values:", exponentiated_sim)

        # Calculate the numerator for positive pairs
        numerator_pos = torch.exp(p_pairs_sim/temperature)
        # Calculate InfoNCE loss
        loss = -torch.log(numerator_pos / denominator)
        loss = loss.mean()

        return loss


#Updated SupConLoss
class SupConLoss(nn.Module):
    """Supervised Contrastive Learning: https://arxiv.org/pdf/2004.11362.pdf.
    It also supports the unsupervised contrastive loss in SimCLR"""
    def __init__(self, temperature=0.07, contrast_mode='all',
                 base_temperature=0.4):
        super(SupConLoss, self).__init__()
        self.temperature = temperature
        self.contrast_mode = contrast_mode
        self.base_temperature = base_temperature

    def forward(self, features, labels=None, mask=None):
        """Compute loss for model. If both `labels` and `mask` are None,
        it degenerates to SimCLR unsupervised loss:
        https://arxiv.org/pdf/2002.05709.pdf

        Args:
            features: hidden vector of shape [bsz, n_views, ...].
            labels: ground truth of shape [bsz].
            mask: contrastive mask of shape [bsz, bsz], mask_{i,j}=1 if sample j
                has the same class as sample i. Can be asymmetric.
        Returns:
            A loss scalar.
        # """
        # device = (torch.device('cuda')
        #           if features.is_cuda
        #           else torch.device('cpu'))
        if len(features.shape) < 3:
                        raise ValueError('`features` needs to be [bsz, n_views, ...],'
                             'at least 3 dimensions are required')
        if len(features.shape) > 3:
            features = features.view(features.shape[0], features.shape[1], -1)

        batch_size = features.shape[0]
        if labels is not None and mask is not None:
            raise ValueError('Cannot define both `labels` and `mask`')
        elif labels is None and mask is None:
            mask = torch.eye(batch_size, dtype=torch.float32).to(CFG.device)
        elif labels is not None:
            labels = labels.contiguous().view(-1, 1)
            # print("Labels:", labels)
            if labels.shape[0] != batch_size:
                raise ValueError('Num of labels does not match num of features')
            mask = torch.eq(labels, labels.T).float().to(CFG.device)
            # print("Mask:", mask)
        else:
            mask = mask.float().to(CFG.device)

        contrast_count = features.shape[1]
          # print("Contrast count:", contrast_count)
        contrast_feature = torch.cat(torch.unbind(features, dim=1), dim=0)
        if self.contrast_mode == 'one':
            anchor_feature = features[:, 0]
            anchor_count = 1
        elif self.contrast_mode == 'all':
            anchor_feature = contrast_feature
            anchor_count = contrast_count
        else:
            raise ValueError('Unknown mode: {}'.format(self.contrast_mode))
        anchor_dot_contrast = torch.div(
            torch.matmul(anchor_feature, contrast_feature.T),
            self.temperature)
        # print("After dot product and dividing by temperature:", anchor_dot_contrast)

        # # for numerical stability
        # logits_max, _ = torch.max(anchor_dot_contrast, dim=1, keepdim=True)
        # logits = anchor_dot_contrast - logits_max.detach()

        logits = anchor_dot_contrast

        # print("Logits Max", logits_max)
        # print("Logits:", logits)

        # tile mask according to n_views
        mask = mask.repeat(anchor_count, contrast_count)

        # # mask-out self-contrast cases
        # logits_mask = torch.scatter(
        #     torch.ones_like(mask),
        #     1,
        #     torch.arange(batch_size * anchor_count).view(-1, 1).to(CFG.device),
        #     0
        # )
        # mask = mask * logits_mask

        upper_triangular_mask = torch.triu(torch.ones_like(mask), diagonal=1).to(CFG.device)
        logits_mask = upper_triangular_mask
        mask = mask * logits_mask


        # compute log_prob
        exp_logits = torch.exp(logits) * logits_mask

        # # Add a small epsilon to prevent log(0)
        epsilon = 1e-12  # Small value to ensure numerical stability
        # denominator = exp_logits.sum(1, keepdim=True) + epsilon  # Add epsilon to avoid zero
        # log_prob = logits - torch.log(denominator)

        global_denominator = exp_logits.sum() + epsilon  # Sum of all elements in the batch

        # Compute log probabilities using the global denominator
        log_prob = logits - torch.log(global_denominator)


        # mask_pos_pairs = mask.sum(1)
        # mask_pos_pairs = torch.where(mask_pos_pairs < 1e-6, 1, mask_pos_pairs)
        # # print("mask_pos_pairs:", mask_pos_pairs)
        # mean_log_prob_pos = (mask * log_prob).sum(1) / mask_pos_pairs
        # # print("mean_log_prob_pos", mean_log_prob_pos)

        sum_pos = (mask * log_prob)

        loss = - (self.temperature / self.base_temperature) * sum_pos
        non_zero_mask = loss != 0
        non_zero_loss = loss[non_zero_mask]
        loss = non_zero_loss.mean()

        return loss


# #Original SupConLoss
# class SupConLoss(nn.Module):
#     """Supervised Contrastive Learning: https://arxiv.org/pdf/2004.11362.pdf.
#     It also supports the unsupervised contrastive loss in SimCLR"""
#     def __init__(self, temperature=0.07, contrast_mode='all',
#                  base_temperature=0.07):
#         super(SupConLoss, self).__init__()
#         self.temperature = temperature
#         self.contrast_mode = contrast_mode
#         self.base_temperature = base_temperature

    # def forward(self, features, labels=None, mask=None):
    #     """Compute loss for model. If both `labels` and `mask` are None,
    #     it degenerates to SimCLR unsupervised loss:
    #     https://arxiv.org/pdf/2002.05709.pdf

    #     Args:
    #         features: hidden vector of shape [bsz, n_views, ...].
    #         labels: ground truth of shape [bsz].
    #         mask: contrastive mask of shape [bsz, bsz], mask_{i,j}=1 if sample j
    #             has the same class as sample i. Can be asymmetric.
    #     # Returns:
    #     #     A loss scalar.
    #     # # """
    #     # device = (torch.device('cuda')
    #     #           if features.is_cuda
    #     #           else torch.device('cpu'))
    #     if len(features.shape) < 3:
    #         raise ValueError('`features` needs to be [bsz, n_views, ...],'
    #                          'at least 3 dimensions are required')
    #     if len(features.shape) > 3:
    #         features = features.view(features.shape[0], features.shape[1], -1)

        # batch_size = features.shape[0]
        # if labels is not None and mask is not None:
        #     raise ValueError('Cannot define both `labels` and `mask`')
        # elif labels is None and mask is None:
        #     mask = torch.eye(batch_size, dtype=torch.float32).to(CFG.device)
        # elif labels is not None:
        #     labels = labels.contiguous().view(-1, 1)
        #     # print("Labels:", labels)
        #     if labels.shape[0] != batch_size:
        #         raise ValueError('Num of labels does not match num of features')
        #     mask = torch.eq(labels, labels.T).float().to(CFG.device)
        #     # print("Mask:", mask)
        # else:
        #     mask = mask.float().to(CFG.device)

       #  contrast_count = features.shape[1]
       #  # print("Contrast count:", contrast_count)
       #  contrast_feature = torch.cat(torch.unbind(features, dim=1), dim=0)
       #  if self.contrast_mode == 'one':
       #      anchor_feature = features[:, 0]
       #      anchor_count = 1
       #  elif self.contrast_mode == 'all':
       #      anchor_feature = contrast_feature
       #      anchor_count = contrast_count
       #  else:
       #      raise ValueError('Unknown mode: {}'.format(self.contrast_mode))

       #  anchor_dot_contrast = torch.div(
       #      torch.matmul(anchor_feature, contrast_feature.T),
       #      self.temperature)
       # # print("After dot product and dividing by temperature:", anchor_dot_contrast)

       #  # for numerical stability
       #  logits_max, _ = torch.max(anchor_dot_contrast, dim=1, keepdim=True)
       #  logits = anchor_dot_contrast - logits_max.detach()


        # # tile mask according to n_views
        # mask = mask.repeat(anchor_count, contrast_count)
        # # print("Mask:", mask)

        # # mask-out self-contrast cases
        # logits_mask = torch.scatter(
        #     torch.ones_like(mask),
        #     1,
        #     torch.arange(batch_size * anchor_count).view(-1, 1).to(CFG.device),
        #     0
        # )
        # mask = mask * logits_mask

        # # compute log_prob
        # exp_logits = torch.exp(logits) * logits_mask

        # # # Add a small epsilon to prevent log(0)
        # epsilon = 1e-12  # Small value to ensure numerical stability
        # denominator = exp_logits.sum(1, keepdim=True)
        # log_prob = logits - torch.log(denominator)
        # mask_pos_pairs = mask.sum(1)
        # mask_pos_pairs = torch.where(mask_pos_pairs < 1e-6, 1, mask_pos_pairs)
        # # print("mask_pos_pairs:", mask_pos_pairs)
        # mean_log_prob_pos = (mask * log_prob).sum(1) / mask_pos_pairs
        # # print("mean_log_prob_pos", mean_log_prob_pos)

        # # loss
        # loss = - (self.temperature / self.base_temperature) * mean_log_prob_pos
        # # print("Loss:", loss)
        # loss = loss.view(anchor_count, batch_size).mean()
        # # print("Loss:", loss)

        # return loss

## Initialize Model

In [ ]:
from sklearn.decomposition import PCA

class Identity(nn.Module):
    def __init__(self):
        super(Identity, self).__init__()

    def forward(self, x):
        return x

class MyModel(nn.Module):

   def __init__(
        self,
        temperature=CFG.temperature,
        image_embedding=CFG.image_embedding,
        text_embedding=CFG.text_embedding
    ):
        super().__init__()
        self.image_encoder = ImageEncoder()
        self.text_encoder = TextEncoder()
#         self.temperature = temperature
        self.image_projection = ProjectionHead(embedding_dim=image_embedding)
        self.text_projection = ProjectionHead(embedding_dim=text_embedding)
#         self.logit_scale = torch.ones([]) * -0.2
        self.first_run = True
        self.dropout = nn.Dropout(0.2)

   def forward(self, batch):

        input_ids = batch["input_ids"]
        attention_mask = batch["attention_mask"]
        token_type_ids = batch["token_type_ids"]
        image = batch["image"]

#         image_embeddings = self.image_encoder(image)

#         print("image_features shape:", image_embeddings.shape)
        # model_resnet.fc = Identity()
#         image_embeddings = model_resnet(image)

        # text_embeddings = self.text_encoder(input_ids=input_ids, attention_mask=attention_mask)
#         text_embeddings =  self.text_encoder(ids=input_ids, mask=attention_mask, token_type_ids=token_type_ids)

#         text_embeddings =  self.text_encoder(input_ids, attention_mask, token_type_ids)

        # Check if it's the first run for the text encoder
        if self.first_run:
            # Set the text encoder to eval mode for consistent embeddings
            self.text_encoder.model.eval()
            self.image_encoder.model.eval()
            text_embeddings = self.text_encoder(input_ids, attention_mask, token_type_ids)
            image_embeddings = self.image_encoder(image)
#             print("Initial image_embeddings:", image_embeddings)
            self.first_run = False  # Set the flag to False after the first run

        else:
            self.text_encoder.model.train()
            self.image_encoder.model.train()
            text_embeddings =  self.text_encoder(input_ids, attention_mask, token_type_ids)
            image_embeddings = self.image_encoder(image)
#             print("image_embeddings:", image_embeddings)



        # Getting Image and Text Embeddings (with same dimension)
        image_embeddings = self.image_projection(image_embeddings)
        text_embeddings = self.text_projection(text_embeddings)

        # Apply dropout
        image_embeddings = self.dropout(image_embeddings)
        text_embeddings = self.dropout(text_embeddings)



        #------------------------------------------------------------------#
# #         #Concatenation
#         features = torch.cat((image_embeddings, text_embeddings), dim=1)
#         feature_embedding = torch.nn.functional.normalize(features, p=2, dim=1)

        #--------------------------------------------------------------------#

        feature_embedding = image_embeddings * text_embeddings
        feature_embedding = torch.nn.functional.normalize(feature_embedding, p=2, dim=1)

        #--------------------------------------------------------------------#

        #   #Another Aggregation function
        # Z_con = image_embeddings * text_embeddings
        # Z_com = image_embeddings + text_embeddings

        # feature_embedding = Z_con + Z_com

        # feature_embedding = torch.nn.functional.normalize(feature_embedding, p=2, dim=1)


        return feature_embedding

model = MyModel().to(CFG.device)

## Training Loop Without CLIP

In [ ]:
# Function to convert model's parameters to FP32 format
# from torch.optim.lr_scheduler import ReduceLROnPlateau
#This is done so that our model loads in the provided memory.
def convert_models_to_fp32(model):
    for p in model.parameters():
        p.data = p.data.float()
        p.grad.data = p.grad.data.float()

params = [
        {"params": model.image_encoder.parameters(), "lr": CFG.image_encoder_lr},
        {"params": model.text_encoder.parameters(), "lr": CFG.text_encoder_lr}
    ]


optimizer = torch.optim.AdamW(params, weight_decay= 0.01)

# optimizer = torch.optim.RMSprop(params)
lr_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
      optimizer, mode="min", patience=CFG.patience, factor=CFG.factor
  )
num_epochs = 1
max_grad_norm = 1.0


best_loss = float('inf')  # Initialize with a high value

train_features = None

for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    train_loss = 0
    cos_sim = None
    # margin=1.0
    pbar = tqdm(train_dataloader, total=len(train_dataloader))

    # Iterate through the batches in the training data
    for batch in pbar:
        optimizer.zero_grad()  # Zero out gradients for the optimizer
        batch = {k: v.to(CFG.device) for k, v in batch.items() if k != "caption"}

        # Forward pass through the model
        features = model(batch)
        features = features.unsqueeze(1)

#         print("features:", features.shape)

        # cos_sim = features @ features.t()

        labels = batch["label"].to(CFG.device)


        # total_loss = loss_func(features, labels)
        # total_loss = calculate_loss(cos_sim, labels)
        criterion = SupConLoss(temperature=0.4)
        total_loss = criterion(features, labels)

        torch.cuda.empty_cache()

        total_loss.backward()

        # convert_models_to_fp32(modified_clip)
        optimizer.step()
        train_loss += total_loss
        # print("loss:", total_loss)
        pbar.set_description(f"Epoch {epoch}/{num_epochs}, Loss: {total_loss.item():.4f}")
    # Calculate average training loss for the epoch
    train_loss /= len(train_dataloader)



    model.eval()
    val_loss = 0

    with torch.no_grad():
        for batch in val_dataloader:
            # Forward pass through the model
            batch = {k: v.to(CFG.device) for k, v in batch.items() if k != "caption"}

            # Forward pass through the model
            features = model(batch)
            features = features.unsqueeze(1)

            # cos_sim_val = features @ features.t()

            labels = batch["label"].to(CFG.device)

            # val_loss += calculate_loss(cos_sim_val, labels)
            # val_loss += loss_func(features, labels)
            criterion = SupConLoss(temperature=0.4)
            val_loss += criterion(features, labels)

            # criterion = CustomContrastiveLoss(temperature=0.4)
            # val_loss += criterion(features, labels)

            torch.cuda.empty_cache()

    # Calculate average validation loss for the epoch
    val_loss /= len(val_dataloader)
    # history["val_loss_existing"].append(val_loss)

    # lr_scheduler.step(val_loss)

    # # Print and/or log the training and validation losses for the epoch
    print(f"Epoch {epoch+1}/{num_epochs} - Training Loss: {train_loss:.4f} - Validation Loss: {val_loss:.4f}")


# torch.save(model.state_dict(), "/content/drive/MyDrive/Experiments/Hate detection with MyModel/Model1_orig(15)_finetuned.pt")

  0%|          | 0/563 [00:00<?, ?it/s]<ipython-input-10-d9bf32b53a91>:27: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  key: torch.tensor(values[idx])
Epoch 0/1, Loss: 4.7813:  53%|█████▎    | 301/563 [04:41<04:04,  1.07it/s]


KeyboardInterrupt: 

# Load Trained Model

In [ ]:
# # Load the saved model state_dict
# model_path = "/kaggle/working/Model_MAMI_bert_vit_14.pt"
# model= MyModel().to(CFG.device)
# model.load_state_dict(torch.load(model_path, map_location=CFG.device))

<ipython-input-16-aa6d489cb385>:4: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  modified_clip.load_state_dict(torch.load(model_path, map_location=CFG.device))


FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/working/Model_MAMI_bert_vit_14.pt'

# Testing

## Test 7 (Using KNN)

In [ ]:
test_labels = pd.read_csv("/kaggle/input/mami-dataset/test_labels.txt", sep='\t', header=None)
second_column = test_labels.iloc[:, 1]
df_test = pd.read_csv(CFG.test_file)

df_test['misogynous'] = second_column
df_test

,file_name,transcripts,misogynous
0,15236.jpg,facebook singles groups belike when a new woma...,0
1,15805.jpg,"so, if you are a feminist how can you eat dairy?",1
2,16254.jpg,when a cute girl left your message on seen,0
3,16191.jpg,photographing something you want to show every...,1
4,15952.jpg,hey babe can you make me a sandwich? hey babe ...,0
...,...,...,...
995,15591.jpg,it is not your fault you did not design the di...,1
996,15049.jpg,think about how much better her skin is breath...,0
997,15363.jpg,the stereotypes are true f she does have a tig...,1
998,15199.jpg,draws naked pictures of black women 00 0000 ge...,0


In [ ]:
test_image_names = df_test['file_name']
test_image_path = []
for name in test_image_names:
  test_image_path.append(CFG.image_path + name)

test_texts =  df_test['transcripts']
test_texts = test_texts.astype(str)
test_labels =  df_test['misogynous']

test_dataloader = build_loaders(df_test, bert_tokenizer,feature_extractor, mode="valid")

In [ ]:
model.eval()  # Set the model to evaluation mode
model.text_encoder.model.eval()
model.image_encoder.model.eval()
test_embeddings = []

test_loss = 0
correct = 0
total = 0
# Disable gradient calculation for testing
pbar = tqdm(test_dataloader, total=len(test_dataloader))
with torch.no_grad():
  for batch in pbar:
        # print(feature_embedding.shape)
        batch = {k: v.to(CFG.device) for k, v in batch.items() if k != "caption"}
        # Forward pass through the model
        test_embedding = model(batch)
        test_embeddings.append(test_embedding)

test_embeddings_list = torch.cat(test_embeddings, dim=0)

  0%|          | 0/63 [00:00<?, ?it/s]<ipython-input-11-dfad699e0f67>:27: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  key: torch.tensor(values[idx])
100%|██████████| 63/63 [00:21<00:00,  2.89it/s]


In [ ]:
model.eval()  # Set the model to evaluation mode
model.text_encoder.model.eval()
model.image_encoder.model.eval()
rtrain_embeddings = []


pbar = tqdm(train_dataloader, total=len(train_dataloader))
with torch.no_grad():
  for batch in pbar:

#         rtrain_embeddings.append(rtrain_embedding)
        batch = {k: v.to(CFG.device) for k, v in batch.items() if k != "caption"}
        # Forward pass through the model
        rtrain_embedding = model(batch)
        rtrain_embeddings.append(rtrain_embedding)


rtrain_embeddings_list = torch.cat(rtrain_embeddings, dim=0)

t_cos_sim = test_embeddings_list @ rtrain_embeddings_list.t()

print(t_cos_sim)

  0%|          | 0/563 [00:00<?, ?it/s]<ipython-input-11-dfad699e0f67>:27: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  key: torch.tensor(values[idx])
100%|██████████| 563/563 [01:57<00:00,  4.80it/s]

tensor([[0.4579, 0.1325, 0.1747,  ..., 0.2486, 0.3796, 0.4099],
        [0.4761, 0.1359, 0.2573,  ..., 0.1895, 0.5067, 0.4912],
        [0.3392, 0.1995, 0.2423,  ..., 0.1939, 0.3089, 0.3726],
        ...,
        [0.4857, 0.1545, 0.2181,  ..., 0.1838, 0.4904, 0.5811],
        [0.4307, 0.1546, 0.2816,  ..., 0.2038, 0.3534, 0.3259],
        [0.2118, 0.3075, 0.4186,  ..., 0.4813, 0.1028, 0.1979]],
       device='cuda:0')


In [ ]:
import torch

# Calculate the dot product between test embeddings and train embeddings
# dot_product = test_embeddings_list @ rtrain_embeddings_list.t()

k = 11
# Find the k highest values and their corresponding indices for each row
k_highest_values, k_highest_indices = torch.topk(t_cos_sim, k, dim=1)

# Move the k highest indices tensor to the CPU
k_highest_indices = k_highest_indices.cpu()

# Find the most frequent label for each test sample
predicted_labels = []
for indices in k_highest_indices:
    labels = train_labels.iloc[indices]
#     print("labels:", labels)
    unique_labels, counts = np.unique(labels, return_counts=True)
#     print("unique labels", unique_labels)
    most_frequent_label = unique_labels[np.argmax(counts)]
    predicted_labels.append(most_frequent_label)

# print("Predicted labels:", predicted_labels)

Predicted labels: [1, 1, 1, 1, 1, 0, 1, 0, 1, 1, 0, 1, 1, 0, 1, 0, 1, 0, 1, 0, 1, 1, 0, 0, 0, 0, 1, 1, 0, 0, 1, 0, 0, 0, 1, 1, 1, 0, 0, 1, 1, 0, 0, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 0, 0, 1, 0, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 0, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 0, 1, 1, 0, 1, 1, 1, 1, 1, 0, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 0, 1, 1, 1, 0, 1, 1, 1, 1, 0, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 1, 1, 1, 0, 1, 1, 1, 0, 0, 1, 1, 1, 1, 1, 0, 0, 0, 1, 0, 1, 0, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 1, 1, 0, 0, 1, 0, 1, 0, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 0, 0, 0, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 0, 1, 0, 1, 1, 1, 0, 1, 1, 0, 1, 1, 0, 0, 1, 0, 0, 1, 1, 1, 0, 1, 1, 1, 1, 0, 0, 1, 1, 0, 1, 0, 0, 0, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 0, 1, 0, 1, 0, 1, 1, 1, 1, 0, 0, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 0, 1, 0, 1, 1, 1, 0, 1, 1, 0, 1, 0, 0, 0, 1, 0, 1, 0, 0, 1, 1, 1, 1, 1, 0, 1, 0, 1, 1, 1, 1, 0, 1, 1, 1, 0, 1, 0, 0, 1, 1, 1, 

## Evaluation Matrix

In [ ]:
test_l = test_labels.values.tolist()
test_labels.shape

(1000,)

In [ ]:
# Count the number of correct 1 labels
correct_1_labels = sum(pred == orig == 1 for pred, orig in zip(predicted_labels, test_l))

# Count the number of correct 0 labels
correct_0_labels = sum(pred == orig == 0 for pred, orig in zip(predicted_labels, test_l))

# Print the results
print("Number of correct 1 labels:", correct_1_labels)
print("Number of correct 0 labels:", correct_0_labels)

Number of correct 1 labels: 454
Number of correct 0 labels: 243


In [ ]:
target_names = ['0', '1']
print(classification_report(test_l, predicted_labels, target_names=target_names), '\n')

              precision    recall  f1-score   support

           0       0.84      0.49      0.62       500
           1       0.64      0.91      0.75       500

    accuracy                           0.70      1000
   macro avg       0.74      0.70      0.68      1000
weighted avg       0.74      0.70      0.68      1000
 



In [ ]:
from sklearn.metrics import f1_score
# macro_f1 = f1_score(test_labels, predicted_labels, average='macro')
macro_f1 = f1_score(test_labels, predicted_labels)

macro_f1

### Confusion Matrix

In [ ]:
from sklearn.metrics import roc_auc_score
from sklearn.metrics import roc_curve
from sklearn.metrics import f1_score
from sklearn.metrics import confusion_matrix

# Compute the confusion matrix
cm = confusion_matrix(test_labels, predicted_labels)

cm_df = pd.DataFrame(cm, index=['True_0', 'True_1'], columns=['Pred_0', 'Pred_1'])

# Print the confusion matrix
print("Confusion Matrix:")
print(cm_df)

auc = roc_auc_score(test_labels, predicted_labels)
print("Area under curve:", auc )
f1s = f1_score(test_labels, predicted_labels)
print(f1s)

Confusion Matrix:
        Pred_0  Pred_1
True_0     195     305
True_1      38     462
Area under curve: 0.657
0.7292817679558011
